# 05 — Algorithms under Objective v3 (Phase 1)

Phase 0 (`04_objective_v3`) checked that the blind objective

$$f(s) = \sum_{k=1}^{K} \big(S(t_k, v_k) - \tau\big), \qquad K \text{ free}$$

rewards the real reflections and penalizes noise. Phase 1 runs the
three algorithms under it — **Random Search, Hill Climbing and Tabu
Search** — on the main case.

**What changed in the optimizers** (all switched on by
`objective='threshold'`; the default `'mean'` objective reproduces the
v1/v2 runs bit-for-bit):
- **Variable number of picks**: solutions have $k_{min} \le K \le k_{max}$
  picks; random solutions also draw K.
- **Insert / remove moves**: with probabilities `p_insert` / `p_remove`
  a neighbor adds a pick in a free gap or removes one (both valid by
  construction). Otherwise the usual moves apply (shift; jump in Tabu
  Search).
- **Tabu attributes by time**: with insertions and removals, positions
  in the list are no longer stable, so Tabu Search forbids "putting a
  pick back at a time it just left" and "removing a pick just inserted".

**How results are scored** (ground truth, evaluation only):
- **Precision** — fraction of picks that match a natural event (within
  `TIME_TOL_MS`, with a correct velocity): how many picks are real.
- **Recall** — fraction of natural events that received a matching pick:
  how many events were found.
- The v3 objective values are sums, on a different scale from the v2
  mean semblance: they are compared with the **reference solution**
  (one pick on each natural event, at the true velocity), not with v2.

This is a single-run visual check per algorithm, not the statistical
protocol.

**Execution order:**
1. Setup
2. Parameters
3. Load the case and compute the blind threshold
4. Reference solution (validation)
5. Run the three algorithms
6. Visualize the picks
7. Convergence
8. Metrics
9. Save results
10. Summary


## 1. Setup

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# Project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')


In [ ]:
from src.seismic.cases     import load_case
from src.seismic.semblance import (compute_semblance, robust_threshold,
                                   semblance_along_curve,
                                   picking_objective_threshold)
from src.analysis.events   import select_events
from src.analysis.metrics  import velocity_errors, precision_recall
from src.optimizers.random_search import RandomSearch
from src.optimizers.hill_climbing import HillClimbing
from src.optimizers.tabu_search   import TabuSearch

print('All modules imported successfully.')


## 2. Parameters

In [ ]:
# --- Case ---
CASE         = 'marmousi2_cdp06800'

# --- Blind formulation v3 (same as 04_objective_v3) ---
VEL_MIN      = 1400.0        # water velocity as the floor
VEL_MAX      = 5000.0        # generous ceiling
VEL_STEP     = 20.0          # velocity increment of the panel used for tau
MIN_SEP_MS   = 100.0         # minimum separation between picks
TAU_FACTOR   = 10.0          # tau = median + TAU_FACTOR * sigma_rob of the panel
# Time window: the whole record. K is free (1 .. as many as fit).

# --- Moves that change the number of picks (all algorithms) ---
P_INSERT     = 0.1           # probability that a neighbor inserts a pick
P_REMOVE     = 0.1           # probability that a neighbor removes a pick

# --- Budget (same for every algorithm) ---
MAX_EVALS    = 100000
SEED         = 42

# --- Local moves (Hill Climbing and Tabu Search) ---
STEP_TIME    = 5             # max time perturbation (samples)
STEP_VEL     = 50.0          # max velocity perturbation (m/s)
N_NEIGHBORS  = 30            # neighbors evaluated per iteration
MAX_ITER     = 3 * MAX_EVALS // N_NEIGHBORS   # upper bound only

# --- Hill Climbing ---
HC_PATIENCE  = 60            # iterations without improvement before a restart
HC_RESTARTS  = 10**6         # restart until the budget is used up

# --- Tabu Search ---
TABU_TENURE  = 15
P_JUMP       = 0.1
TS_PATIENCE  = MAX_ITER      # no early stop: the budget is the stopping rule

# --- Evaluation only (ground truth) ---
CORRECT_TOL  = 0.02          # relative velocity tolerance
TIME_TOL_MS  = 40.0          # pick-event time tolerance: covers the 30 ms
                              # semblance gate plus the wavelet, and stays
                              # below MIN_SEP_MS / 2 so matches are unique
EARLY_MS     = 100.0         # picks before this time are reported apart

print('Parameters set.')


## 3. Load the case and compute the blind threshold

Only the gather is used here: the panel over the v3 velocity range and
the threshold computed from it.

In [ ]:
case   = load_case(os.path.join(project_root, 'cases', f'{CASE}.npz'))
gather, offsets, dt = case['gather'], case['offsets'], case['dt_ms']
t_grid, v_rms_true  = case['t_grid'], case['v_rms_true']

velocities = np.arange(VEL_MIN, VEL_MAX + VEL_STEP, VEL_STEP, dtype=np.float32)
panel = compute_semblance(gather, offsets, velocities, dt)
TAU   = robust_threshold(panel, factor=TAU_FACTOR)

print(f'Case     : {CASE}')
print(f'Gather   : {gather.shape}, record {t_grid[-1]:.0f} ms')
print(f'tau      : {TAU:.3f}  (median + {TAU_FACTOR:g} robust spreads of the panel)')


## 4. Reference solution (validation)

One pick on each natural event (local maxima of semblance along the
true V_rms, above $\tau$, at least `MIN_SEP_MS` apart), at the true
velocity. Its objective value $f_{ref}$ is a yardstick for the
algorithms. It is not necessarily the maximum of $f$: at each time the
optimizer may find a slightly higher semblance at a velocity or time a
little off the true curve.

In [ ]:
s_true   = semblance_along_curve(gather, offsets, v_rms_true, dt)
min_sep  = int(round(MIN_SEP_MS / dt))
events   = select_events(s_true, TAU, min_sep)
event_ms = events * dt
reference = [(int(i), float(v_rms_true[i])) for i in events]
f_ref     = picking_objective_threshold(gather, offsets, reference, dt, TAU)

print(f'Natural events : {len(events)}')
print(f'f_ref          : {f_ref:.3f}')


## 5. Run the three algorithms

In [ ]:
common = dict(
    traces=gather, offsets=offsets, dt_ms=dt,
    vel_min=VEL_MIN, vel_max=VEL_MAX, min_sep_ms=MIN_SEP_MS,
    objective='threshold', tau=TAU, p_insert=P_INSERT, p_remove=P_REMOVE,
    seed=SEED, max_evals=MAX_EVALS,
)
local = dict(step_time=STEP_TIME, step_vel=STEP_VEL,
             n_neighbors=N_NEIGHBORS, max_iter=MAX_ITER)

algorithms = {
    'Random Search': lambda: RandomSearch(**common),
    'Hill Climbing': lambda: HillClimbing(**common, **local,
                                          restarts=HC_RESTARTS,
                                          patience=HC_PATIENCE),
    'Tabu Search':   lambda: TabuSearch(**common, **local,
                                        tabu_tenure=TABU_TENURE,
                                        patience=TS_PATIENCE, p_jump=P_JUMP),
}

results = {}
for name, make in algorithms.items():
    print(f'Running {name}...')
    opt = make().run()
    t_s, v_s = opt.get_result()
    results[name] = {'opt': opt, 'times_ms': t_s * dt, 'vels': v_s}
    print(f'   f = {opt.best_score:.3f}   K = {len(t_s)}   '
          f'evals = {opt.n_evals}   time = {opt.exec_time_s:.1f} s')

ts = results['Tabu Search']['opt']
print()
print(f'Tabu Search accepted moves: {ts.move_stats}')
print(f'Hill Climbing restarts    : {len(results["Hill Climbing"]["opt"].restart_scores)}')


## 6. Visualize the picks

In [ ]:
os.makedirs(os.path.join(project_root, 'results', 'figures'), exist_ok=True)

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 8), sharey=True)

for ax, (name, r) in zip(axes, results.items()):
    pr = precision_recall(r['times_ms'], r['vels'], t_grid, v_rms_true,
                          event_ms, tol=CORRECT_TOL, time_tol_ms=TIME_TOL_MS)
    matched = np.array(pr['matched'], dtype=bool)

    ax.imshow(panel.T, aspect='auto', cmap='jet', vmin=0, vmax=1,
              extent=[velocities[0], velocities[-1], t_grid[-1], t_grid[0]])
    ax.plot(v_rms_true, t_grid, 'w--', linewidth=1.2, label='True V RMS')
    ax.plot(v_rms_true[events], event_ms, 'o', mfc='none', mec='white',
            markersize=10, mew=1.2, label='Natural events')
    ax.plot(r['vels'], r['times_ms'], 'w-', linewidth=0.8, alpha=0.6)
    ax.plot(r['vels'][matched], r['times_ms'][matched], 'o', color='black',
            markeredgecolor='white', markersize=6, label='Pick (matches an event)')
    ax.plot(r['vels'][~matched], r['times_ms'][~matched], 'X', color='red',
            markeredgecolor='white', markersize=9, label='Pick (no match)')
    ax.set_xlim(VEL_MIN, 3500)
    ax.set_xlabel('Velocity (m/s)')
    ax.set_title(f"{name}\nK = {pr['n_picks']}, precision {pr['precision']:.0%}, "
                 f"recall {pr['recall']:.0%}", fontsize=11)

axes[0].set_ylabel('Time (ms)')
axes[0].legend(loc='lower left', fontsize=8)
plt.suptitle(f'Objective v3 (blind) — {CASE} — {MAX_EVALS} evaluations each '
             f'(panel shown up to 3500 m/s; search range {VEL_MIN:.0f}–{VEL_MAX:.0f} m/s)',
             fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures',
                         '05_v3_algorithms_picks.png'), dpi=150)
plt.show()
print('Figure saved.')


## 7. Convergence

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, r in results.items():
    hist = r['opt'].eval_history          # (evaluations, best f) at each improvement
    x = [e for e, _ in hist] + [r['opt'].n_evals]
    y = [f for _, f in hist] + [hist[-1][1]]
    ax.step(x, y, where='post', linewidth=1.5, label=name)

ax.axhline(f_ref, color='black', linestyle='--', linewidth=1.0,
           label=f'Reference solution (f = {f_ref:.2f})')
ax.set_xlabel('Objective evaluations')
ax.set_ylabel('Best f found')
ax.set_title('Best objective value vs. evaluations')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures',
                         '05_v3_algorithms_convergence.png'), dpi=150)
plt.show()
print('Figure saved.')


## 8. Metrics

In [ ]:
rows = {}
for name, r in results.items():
    pr  = precision_recall(r['times_ms'], r['vels'], t_grid, v_rms_true,
                           event_ms, tol=CORRECT_TOL, time_tol_ms=TIME_TOL_MS)
    err = velocity_errors(r['times_ms'], r['vels'], t_grid, v_rms_true)
    rows[name] = {
        'f': float(r['opt'].best_score), 'K': pr['n_picks'],
        'vel_correct': pr['n_vel_correct'],
        'precision': pr['precision'], 'recall': pr['recall'], 'f1': pr['f1'],
        'rmse': err['rmse'], 'early_picks': int(np.sum(r['times_ms'] < EARLY_MS)),
        'n_evals': r['opt'].n_evals, 'exec_time_s': float(r['opt'].exec_time_s),
    }

print(f'Reference: K = {len(events)}, f = {f_ref:.3f}')
print()
print(f'{"Algorithm":>14} {"f":>8} {"K":>4} {"vel ok":>7} {"Prec.":>6} {"Recall":>7} '
      f'{"F1":>5} {"RMSE":>7} {"<100ms":>7} {"Time (s)":>9}')
print('-' * 84)
for name, m in rows.items():
    print(f"{name:>14} {m['f']:>8.3f} {m['K']:>4} {m['vel_correct']:>4}/{m['K']:<2} "
          f"{m['precision']:>6.0%} {m['recall']:>7.0%} {m['f1']:>5.2f} "
          f"{m['rmse']:>7.1f} {m['early_picks']:>7} {m['exec_time_s']:>9.1f}")

print()
print('"vel ok": picks with a velocity within 2% of the true V RMS at the pick time,')
print('wherever they are. "<100ms": picks in the first 100 ms of the record, where')
print('shallow trajectories crossing deep events on the far traces produce narrow')
print('semblance spikes (no NMO stretch mute is applied).')


## 9. Save results

In [ ]:
os.makedirs(os.path.join(project_root, 'results', 'metrics'), exist_ok=True)

summary = {
    'case': CASE, 'tau': TAU, 'tau_factor': TAU_FACTOR,
    'formulation': {'vel_min': VEL_MIN, 'vel_max': VEL_MAX,
                    'min_sep_ms': MIN_SEP_MS, 'time_window': 'whole record',
                    'p_insert': P_INSERT, 'p_remove': P_REMOVE},
    'max_evals': MAX_EVALS, 'seed': SEED,
    'reference': {'K': int(len(events)), 'f': f_ref,
                  'event_times_ms': event_ms.tolist()},
    'algorithms': {name: {**rows[name],
                          'times_ms': results[name]['times_ms'].tolist(),
                          'vels': results[name]['vels'].tolist()}
                   for name in results},
    'tabu_move_stats': ts.move_stats,
}

path = os.path.join(project_root, 'results', 'metrics', 'v3_algorithms.json')
with open(path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved: {path}')


## 10. Summary

**What phase 1 shows.** Whether the three algorithms can work with a
free number of picks under the blind v3 objective: how close they get
to the reference solution, how many picks they choose, and how many of
those picks are real (precision) and how many events they find
(recall).

**Points to keep in mind when reading the results.**
- A single run per algorithm: it shows behavior, not typical
  performance. The statistical protocol (paired seeds, all cases) comes
  next.
- The objective can exceed $f_{ref}$: at each time the semblance
  maximum may lie slightly off the true curve (within the semblance
  gate), so the best picks are not always exactly on the reference.
- Precision compares the velocity with V_rms at the **pick** time. A
  pick shifted by a few tens of ms from its event, right where V_rms
  changes quickly (e.g. below the water bottom), can carry the event's
  velocity and still miss the 2% tolerance.
- Picks in the first ~100 ms come from narrow semblance spikes of
  shallow trajectories crossing deep events on the far traces. An NMO
  stretch mute — standard processing, using only the acquisition
  geometry — would remove them; it is not applied yet.

**Next steps.**
- Decide on the NMO stretch mute.
- Statistical protocol under v3: paired seeds, all test cases,
  sensitivity to `TAU_FACTOR` and `MIN_SEP_MS`.
